# Explore municipal waste for the dashboard

Exploration only: anything the dashboard uses must come from a script in `scripts/`.

Data: `data/processed/municipal_waste_decoupling.json` (built by `scripts/processed/municipal_waste_decoupling.py`): municipal waste and real GDP per person, both indexed to 2011 = 100, for the 27 EU Member States and `EU27_2020`. Notes: `data/lookups/decoupling_notes.json` (built by `scripts/lookups/create_decoupling_notes.py`).

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from geo import EU27_TOTAL, geo_names
from paths import DIR_LOOKUPS, DIR_PROCESSED

df_decoupling = pd.read_json(DIR_PROCESSED / "municipal_waste_decoupling.json")
names = geo_names()  # code -> display name, e.g. "BE" -> "Belgium", "EU27_2020" -> "EU27"

# notes shown under the decoupling chart (general note + country notes with break years)
with open(DIR_LOOKUPS / "decoupling_notes.json", encoding="utf-8") as f:
    notes = json.load(f)
country_notes = {n["geo"]: n for n in notes["country_notes"]}

df_decoupling.head()

## Decoupling by country: municipal waste and real GDP per person, 2011 = 100

Choose a country (or the EU27) in the dropdown; the chart and the notes update. This is a prototype of the dashboard interaction.

Both series are indexed to 2011 = 100, so they share one axis. Where the GDP line is above the waste line, the economy grew more than waste per person: waste is decoupled from economic growth.

The notes under the chart come from `data/lookups/decoupling_notes.json`, as in the dashboard: a country note where the data needs care (with a marker on the break year), then the general note.

In [ ]:
import ipywidgets as widgets

WASTE = "#2a78d6"  # categorical slot 1 (blue)
GDP = "#eb6834"    # categorical slot 2 (orange)

# Fixed y axis for all countries, so the same distance always means the same change and
# countries can be compared fairly. Set from the lowest and highest index of all countries,
# rounded out to the nearest 10 (80 to 210 with the current data).
INDEX_COLS = ["mw_pc_index", "gdp_pc_index"]
Y_MIN = int(df_decoupling[INDEX_COLS].min().min() // 10 * 10)
Y_MAX = int(-(-df_decoupling[INDEX_COLS].max().max() // 10) * 10)


def plot_decoupling(geo):
    """Line chart of the waste and GDP indices (2011 = 100) for one geography, with its notes."""
    d = df_decoupling.loc[df_decoupling["geo"] == geo].sort_values("year")
    note = country_notes.get(geo)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(d["year"], d["mw_pc_index"], color=WASTE, linewidth=2.5, label="Municipal waste per person")
    ax.plot(d["year"], d["gdp_pc_index"], color=GDP, linewidth=2.5, label="Real GDP per person")

    # label the last value of each line
    for col, color in [("mw_pc_index", WASTE), ("gdp_pc_index", GDP)]:
        last = d.dropna(subset=[col]).iloc[-1]
        ax.annotate(f"{last[col]:.0f}", (last["year"], last[col]),
                    textcoords="offset points", xytext=(6, 0), va="center", color=color, fontsize=10)

    ax.axhline(100, color="#888888", linewidth=1, linestyle="--")  # base year 2011 = 100

    # mark the year of a break in the series of waste per person
    if note and note["break_year"]:
        ax.axvline(note["break_year"], color="#888888", linewidth=1, linestyle=":")
        ax.annotate(f"Break in series {note['break_year']}", (note["break_year"], 1),
                    xycoords=("data", "axes fraction"), xytext=(4, -4), textcoords="offset points",
                    va="top", fontsize=9, color="#555555")

    ax.set_ylim(Y_MIN, Y_MAX)  # same y axis for every country
    ax.set_title(f"Municipal waste and real GDP per person, {names[geo]} (2011 = 100)", loc="left", fontsize=12)
    ax.set_ylabel("Index, 2011 = 100")
    ax.set_xticks(d["year"])
    ax.tick_params(axis="x", labelrotation=45)
    ax.legend(frameon=False, loc="upper left")
    ax.grid(axis="y", color="#eeeeee")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()

    # notes under the chart: country note (if any), then the general note
    if note:
        print(f"Note for {names[geo]}: {note['note']}\n")
    print(notes["general_note"])


# dropdown: EU27 first, then the 27 Member States in alphabetical order of their names
options = [(names[EU27_TOTAL], EU27_TOTAL)] + sorted(
    ((name, code) for code, name in names.items() if code != EU27_TOTAL)
)
widgets.interact(plot_decoupling, geo=widgets.Dropdown(options=options, value=EU27_TOTAL, description="Country"));